# Quantnuis - Entraînement NoisyCarDetector

Entraîne un Random Forest sur les features audio extraites depuis Google Drive.

**Workflow :**
1. Monte Google Drive (données sync via `gdrive_sync.py`)
2. Extrait les features audio (~180 features par slice)
3. Sélectionne les meilleures features
4. Entraîne un Random Forest + MLP
5. Sauvegarde les artifacts sur Drive

## 1. Setup

In [ ]:
# Monter Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Installer les deps manquantes
!pip install -q librosa soundfile imbalanced-learn joblib

In [ ]:
import os
import numpy as np
import pandas as pd
import librosa
import joblib
import warnings
import time
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import f1_score, classification_report, confusion_matrix, accuracy_score
from imblearn.over_sampling import SMOTE

warnings.filterwarnings('ignore')

# Config
SR = 22050
DURATION = 4.0
N_MFCC = 40

# Chemins Drive
DRIVE_BASE = Path('/content/drive/MyDrive/Quantnuis/noisy_car_detector')
SLICES_DIR = DRIVE_BASE / 'slices'
ANNOTATION_CSV = DRIVE_BASE / 'annotation.csv'
ARTIFACTS_DIR = DRIVE_BASE / 'artifacts'

print(f'Drive monté: {DRIVE_BASE.exists()}')
print(f'Annotation: {ANNOTATION_CSV.exists()}')
if SLICES_DIR.exists():
    n_slices = len(list(SLICES_DIR.glob('*.wav')))
    print(f'Slices: {n_slices}')

## 2. Copie locale (I/O plus rapide que Drive)

In [ ]:
# Copier les slices en local pour un I/O beaucoup plus rapide
LOCAL_DIR = Path('/content/data')
LOCAL_SLICES = LOCAL_DIR / 'slices'

if not LOCAL_SLICES.exists():
    print('Copie des slices depuis Drive vers local (peut prendre quelques minutes)...')
    !mkdir -p /content/data/slices
    !cp {ANNOTATION_CSV} /content/data/
    !cp -r {SLICES_DIR}/* /content/data/slices/
    print(f'Copié: {len(list(LOCAL_SLICES.glob("*.wav")))} fichiers')
else:
    print(f'Données locales déjà présentes: {len(list(LOCAL_SLICES.glob("*.wav")))} fichiers')

## 3. Extraction de features

In [ ]:
def extract_features(filepath):
    """Extrait ~80 features audio d'un fichier WAV."""
    try:
        y, sr = librosa.load(filepath, sr=SR, duration=DURATION)
        if len(y) < SR * 2:  # trop court
            return None

        features = {}

        # MFCC (40 coefficients x mean/std = 80 features)
        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=N_MFCC)
        for i in range(N_MFCC):
            features[f'mfcc_{i}_mean'] = np.mean(mfcc[i])
            features[f'mfcc_{i}_std'] = np.std(mfcc[i])

        # Spectral
        features['spectral_centroid_mean'] = np.mean(librosa.feature.spectral_centroid(y=y, sr=sr))
        features['spectral_bandwidth_mean'] = np.mean(librosa.feature.spectral_bandwidth(y=y, sr=sr))
        features['spectral_rolloff_mean'] = np.mean(librosa.feature.spectral_rolloff(y=y, sr=sr))
        features['spectral_flatness_mean'] = np.mean(librosa.feature.spectral_flatness(y=y))

        # Temporal
        features['rms_mean'] = np.mean(librosa.feature.rms(y=y))
        features['rms_std'] = np.std(librosa.feature.rms(y=y))
        features['zcr_mean'] = np.mean(librosa.feature.zero_crossing_rate(y=y))
        features['zcr_std'] = np.std(librosa.feature.zero_crossing_rate(y=y))

        # Chroma
        chroma = librosa.feature.chroma_stft(y=y, sr=sr)
        for i in range(12):
            features[f'chroma_{i}_mean'] = np.mean(chroma[i])

        return features
    except Exception:
        return None

In [ ]:
# Charger annotations
df_ann = pd.read_csv(LOCAL_DIR / 'annotation.csv')
print(f'Annotations: {len(df_ann)}')
print(f'Label 0 (normal): {(df_ann.label == 0).sum()}')
print(f'Label 1 (bruyant): {(df_ann.label == 1).sum()}')

In [ ]:
# Extraire les features (sequentiel mais efficace sur Colab)
features_path = LOCAL_DIR / 'features.csv'

if features_path.exists():
    print('Features déjà extraites, chargement...')
    df_features = pd.read_csv(features_path)
else:
    print(f'Extraction des features pour {len(df_ann)} fichiers...')
    rows = []
    errors = 0
    t0 = time.time()

    for i, row in df_ann.iterrows():
        fpath = LOCAL_SLICES / row['nfile']
        feats = extract_features(str(fpath))

        if feats:
            feats['nfile'] = row['nfile']
            feats['label'] = int(row['label'])
            rows.append(feats)
        else:
            errors += 1

        if (i + 1) % 1000 == 0:
            elapsed = time.time() - t0
            rate = (i + 1) / elapsed
            eta = (len(df_ann) - i - 1) / rate / 60
            print(f'  [{i+1}/{len(df_ann)}] {rate:.0f} fichiers/s | ETA: {eta:.1f} min | Erreurs: {errors}')

    df_features = pd.DataFrame(rows)
    df_features.to_csv(features_path, index=False)
    elapsed = time.time() - t0
    print(f'\nTerminé: {len(df_features)} features extraites en {elapsed/60:.1f} min ({errors} erreurs)')

In [ ]:
print(f'Dataset: {df_features.shape}')
print(f'Features: {df_features.shape[1] - 2}')  # -nfile, -label
df_features.head()

## 4. Entraînement Random Forest

In [ ]:
# Préparer X, y
meta_cols = ['nfile', 'label']
feature_cols = [c for c in df_features.columns if c not in meta_cols]

X = df_features[feature_cols].values
y = df_features['label'].values

# Nettoyer NaN/Inf
X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

print(f'X: {X.shape}')
print(f'y: {y.shape} | 0={( y==0).sum()} | 1={(y==1).sum()}')

In [ ]:
# Sélection de features via importance RF
N_TOP = 20

rf_selector = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf_selector.fit(X, y)

importance = pd.DataFrame({'feature': feature_cols, 'importance': rf_selector.feature_importances_})
importance = importance.sort_values('importance', ascending=False)

top_features = importance.head(N_TOP)['feature'].tolist()
top_indices = [feature_cols.index(f) for f in top_features]

print(f'Top {N_TOP} features:')
for i, row in importance.head(N_TOP).iterrows():
    print(f"  {row['feature']:30s} {row['importance']:.4f}")

X_sel = X[:, top_indices]

In [ ]:
# Split
X_train, X_test, y_train, y_test = train_test_split(
    X_sel, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {X_train.shape[0]} | Test: {X_test.shape[0]}')

# SMOTE sur train
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)
print(f'Après SMOTE: {X_train_sm.shape[0]} (0={( y_train_sm==0).sum()}, 1={(y_train_sm==1).sum()})')

# Scaler
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train_sm)
X_test_sc = scaler.transform(X_test)

In [ ]:
# Entraîner Random Forest
print('Entraînement Random Forest...')
rf = RandomForestClassifier(
    n_estimators=500,
    max_depth=None,
    min_samples_split=5,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train_sc, y_train_sm)

# Évaluation
y_pred = rf.predict(X_test_sc)
f1 = f1_score(y_test, y_pred)
acc = accuracy_score(y_test, y_pred)

print(f'\n=== Résultats ===')
print(f'Accuracy: {acc*100:.2f}%')
print(f'F1 Score: {f1:.4f}')
print(f'\n{classification_report(y_test, y_pred, target_names=["NORMAL", "BRUYANT"])}')

cm = confusion_matrix(y_test, y_pred)
print(f'Confusion: TN={cm[0,0]} FP={cm[0,1]} FN={cm[1,0]} TP={cm[1,1]}')

In [ ]:
# Cross-validation 5-fold
print('5-Fold Cross-Validation...')
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(rf, X_sel, y, cv=cv, scoring='f1', n_jobs=-1)

print(f'F1 par fold: {cv_scores}')
print(f'F1 moyen: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})')

## 5. Sauvegarde sur Drive

In [ ]:
# Sauvegarder les artifacts sur Drive
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

# Random Forest
rf_path = ARTIFACTS_DIR / 'rf_model.pkl'
joblib.dump(rf, str(rf_path))
print(f'Modèle RF: {rf_path}')

# Scaler
scaler_path = ARTIFACTS_DIR / 'scaler.pkl'
joblib.dump(scaler, str(scaler_path))
print(f'Scaler: {scaler_path}')

# Features
features_txt = ARTIFACTS_DIR / 'features.txt'
with open(features_txt, 'w') as f:
    f.write('\n'.join(top_features))
print(f'Features: {features_txt}')

# Aussi sauver le features.csv sur Drive
df_features.to_csv(DRIVE_BASE / 'features.csv', index=False)
print(f'Features CSV sauvé sur Drive')

print(f'\nTerminé ! F1={f1:.4f} | Accuracy={acc*100:.2f}%')